# 03. Mask-aware Hungarian Matching

## 목표
작은 prediction·ground-truth 집합에서 class, box와 mask cost를 결합하고 모든 일대일 assignment를 탐색합니다. 실제 구현은 Hungarian algorithm을 사용하지만 작은 예제는 permutation으로 최적해를 확인할 수 있습니다.

In [ ]:
from itertools import permutations

# 행은 prediction, 열은 ground truth입니다. 값이 작을수록 좋은 matching입니다.
class_cost = [
    [0.1, 1.2, 0.8],
    [1.0, 0.2, 0.7],
    [0.6, 0.9, 0.2],
]
box_cost = [
    [0.2, 0.8, 0.7],
    [0.7, 0.3, 0.5],
    [0.5, 0.6, 0.4],
]
mask_cost = [
    [0.3, 0.9, 0.8],
    [0.8, 0.7, 0.2],  # prediction 1의 mask는 GT 2와 더 비슷합니다.
    [0.7, 0.2, 0.6],  # prediction 2의 mask는 GT 1과 더 비슷합니다.
]

In [ ]:
def combined_cost(prediction, target, mask_weight):
    return (
        class_cost[prediction][target]
        + box_cost[prediction][target]
        + mask_weight * mask_cost[prediction][target]
    )

def best_assignment(mask_weight):
    candidates = []
    for targets in permutations(range(3)):
        total = sum(combined_cost(pred, target, mask_weight) for pred, target in enumerate(targets))
        candidates.append((total, targets))
    return min(candidates)

for weight in (0.0, 0.5, 1.0, 2.0):
    total, assignment = best_assignment(weight)
    print(f"mask_weight={weight:.1f} assignment={assignment} total={total:.2f}")

## 해석

Mask cost가 없으면 class와 box가 assignment를 결정합니다. Mask weight가 커지면 pixel shape가 더 비슷한 cross assignment가 선택될 수 있습니다. 가중치가 지나치게 크면 class·localization보다 초기의 불안정한 mask를 우선해 학습이 흔들릴 수 있습니다.

## 전문가 확장 과제

1. Dice와 sigmoid focal cost를 별도 행렬로 나누세요.
2. Auxiliary decoder layer별 assignment를 공유할지 다시 계산할지 비교하세요.
3. ROI crop cost와 full-map cost가 다른 assignment를 만드는 반례를 만드세요.
4. 실제 benchmark에서 confidence threshold sweep, PR curve와 latency percentile을 함께 기록하는 평가 schema를 설계하세요.